# 04 · 手刻 Self-Attention

**對應教學 App**：✨ Transformer 與 Attention

用 NumPy 把 Transformer 的核心零件全部實作一次：

1. Scaled Dot-Product Attention
2. 為什麼要除以 √d_k（用實驗證明）
3. Multi-Head Attention
4. Causal Mask（因果遮罩）
5. Positional Encoding（位置編碼）
6. 完整的 Transformer Block

**跑完這份，Transformer 的面試題你就答得出來了。**

---

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams["font.sans-serif"] = ["Microsoft JhengHei", "Microsoft YaHei", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False

np.random.seed(42)
np.set_printoptions(precision=3, suppress=True)
print("✅ 準備完成")

## 1. Softmax：Attention 的關鍵零件

把任意數字轉成「加總為 1 的機率」。

⚠️ **數值穩定的寫法**：一定要先減掉最大值，否則 `exp(1000)` 會溢位變成 `inf`。
面試手寫 softmax 時，漏掉這行是常見扣分點。

In [ ]:
def softmax(x, axis=-1):
    x_max = np.max(x, axis=axis, keepdims=True)
    e = np.exp(x - x_max)                     # ← 減最大值，防溢位
    return e / np.sum(e, axis=axis, keepdims=True)


scores = np.array([2.0, 1.0, 0.1])
print("輸入:", scores)
print("softmax:", softmax(scores))
print("加總:", softmax(scores).sum())
print()
print("溢位測試（不減最大值就會變 nan）:")
print("softmax([1000, 999, 998]) =", softmax(np.array([1000., 999., 998.])))

## 2. Scaled Dot-Product Attention

$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

四個步驟：
1. `QKᵀ` → 每個位置對每個位置的相似度
2. `÷ √d_k` → 縮放
3. `softmax` → 轉成權重（每列加總 = 1）
4. `× V` → 加權平均

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q: (seq_len_q, d_k)
    K: (seq_len_k, d_k)
    V: (seq_len_k, d_v)
    mask: (seq_len_q, seq_len_k)，0 代表要遮掉
    回傳: (輸出, 注意力權重)
    """
    d_k = Q.shape[-1]

    scores = Q @ K.T / np.sqrt(d_k)               # ①②

    if mask is not None:
        scores = np.where(mask == 0, -1e9, scores)  # 遮掉的地方設成極小值

    attn = softmax(scores, axis=-1)               # ③
    output = attn @ V                             # ④
    return output, attn


print("✅ Attention 函式完成 —— Transformer 最核心的 6 行就是這個")

## 3. 實測：用有語意的向量看它在幹嘛

我手工設計 6 個詞的向量，維度是**看得懂的語意軸**。
語意相近的詞，向量也相近 → Attention 應該會讓它們互相關注。

In [ ]:
words = ["貓", "狗", "跑", "跳", "紅色", "藍色"]

#                是動物  是動作  是顏色  移動性
X = np.array([
    [1.0,   0.0,   0.0,   0.3],   # 貓
    [1.0,   0.0,   0.0,   0.4],   # 狗
    [0.0,   1.0,   0.0,   0.9],   # 跑
    [0.0,   1.0,   0.0,   0.8],   # 跳
    [0.0,   0.0,   1.0,   0.0],   # 紅色
    [0.0,   0.0,   1.0,   0.0],   # 藍色
])

out, attn = scaled_dot_product_attention(X, X, X)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(attn, cmap="Blues")
for i in range(len(words)):
    for j in range(len(words)):
        ax.text(j, i, f"{attn[i, j]:.2f}", ha="center", va="center", fontsize=10,
                color="white" if attn[i, j] > attn.max() / 2 else "black")
ax.set_xticks(range(6), words); ax.set_yticks(range(6), words)
ax.set_xlabel("被關注的字（Key）"); ax.set_ylabel("正在看的字（Query）")
ax.set_title("Self-Attention 權重矩陣（每一列加總 = 1）")
plt.colorbar(im); plt.tight_layout(); plt.show()

print("👉 「貓」最關注「狗」，「跑」最關注「跳」，「紅色」最關注「藍色」。")
print("   模型（其實只是內積）自己發現了語意群組。")

## 4. 實驗：證明為什麼要除以 √d_k

**理論**：Q、K 各維度獨立、平均 0 變異數 1 時，
內積 `q·k` 是 d_k 項相加，**變異數 = d_k，標準差 = √d_k**。

我們用隨機數實測看看理論對不對。

In [ ]:
print(f"{'d_k':>6}{'內積的標準差':>16}{'√d_k':>10}{'除以√d_k後':>14}")
print("-" * 48)

for d_k in [4, 16, 64, 256, 1024]:
    q = np.random.randn(5000, d_k)
    k = np.random.randn(5000, d_k)
    dots = np.sum(q * k, axis=1)
    print(f"{d_k:>6}{dots.std():>16.2f}{np.sqrt(d_k):>10.2f}{(dots / np.sqrt(d_k)).std():>14.2f}")

print()
print("👉 內積的標準差確實約等於 √d_k，除完之後穩定回 1。")

In [ ]:
# 看縮放對 softmax 的實際影響
d_k = 64
q = np.random.randn(1, d_k)
K = np.random.randn(8, d_k)

raw = (q @ K.T)[0]
scaled = raw / np.sqrt(d_k)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(range(8), softmax(raw), color="#dc2626")
axes[0].set_title(f"沒縮放：max = {softmax(raw).max():.3f}（幾乎全押一格）")
axes[0].set_ylim(0, 1); axes[0].set_xlabel("位置"); axes[0].set_ylabel("注意力權重")

axes[1].bar(range(8), softmax(scaled), color="#16a34a")
axes[1].set_title(f"除以 √{d_k}：max = {softmax(scaled).max():.3f}（分布平滑）")
axes[1].set_ylim(0, 1); axes[1].set_xlabel("位置")

plt.tight_layout(); plt.show()

print("💡 沒縮放時 softmax 接近 one-hot，而 softmax 在飽和區的梯度趨近 0，")
print("   → 模型學不動。這就是要除以 √d_k 的真正原因。")

## 5. Causal Mask（因果遮罩）

GPT 類模型的訓練目標是「預測下一個字」，
所以**絕對不能讓它看到後面的字**（等於考試給答案）。

做法：把「未來」的位置設成 `-1e9`，softmax 之後就變成 0。

In [ ]:
seq_len = 6
causal_mask = np.tril(np.ones((seq_len, seq_len)))     # 下三角矩陣

print("因果遮罩（1 = 看得到，0 = 遮住）：")
print(causal_mask.astype(int))
print()

X6 = np.random.randn(seq_len, 8)
_, attn_causal = scaled_dot_product_attention(X6, X6, X6, mask=causal_mask)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
_, attn_full = scaled_dot_product_attention(X6, X6, X6)

for ax, a, t in zip(axes, [attn_full, attn_causal],
                    ["沒有遮罩（BERT：前後都看得到）", "因果遮罩（GPT：只看得到前面）"]):
    im = ax.imshow(a, cmap="Blues", vmin=0, vmax=1)
    for i in range(seq_len):
        for j in range(seq_len):
            ax.text(j, i, f"{a[i, j]:.2f}", ha="center", va="center", fontsize=8,
                    color="white" if a[i, j] > .5 else "black")
    ax.set_title(t); ax.set_xlabel("被關注的位置"); ax.set_ylabel("正在看的位置")
    ax.set_xticks(range(seq_len)); ax.set_yticks(range(seq_len))

plt.tight_layout(); plt.show()

print("👉 右圖是完美的下三角形。第 0 個字只能看自己，第 5 個字能看全部前面的。")

## 6. Multi-Head Attention

**重點：不是算 h 次完整 attention**，而是把 d_model 切成 h 份，
各自做 attention 後再接回來。所以**計算量幾乎不變**。

In [ ]:
class MultiHeadAttention:
    def __init__(self, d_model=64, num_heads=8, seed=42):
        assert d_model % num_heads == 0, "d_model 必須能被 num_heads 整除"
        rng = np.random.default_rng(seed)

        self.d_model = d_model
        self.h = num_heads
        self.d_k = d_model // num_heads          # 每個頭分到的維度

        scale = 1 / np.sqrt(d_model)
        self.W_q = rng.normal(0, scale, (d_model, d_model))
        self.W_k = rng.normal(0, scale, (d_model, d_model))
        self.W_v = rng.normal(0, scale, (d_model, d_model))
        self.W_o = rng.normal(0, scale, (d_model, d_model))

    def __call__(self, X, mask=None):
        L = X.shape[0]

        # ① 線性投影成 Q, K, V
        Q, K, V = X @ self.W_q, X @ self.W_k, X @ self.W_v

        # ② 切成 h 個頭：(L, d_model) → (h, L, d_k)
        Q = Q.reshape(L, self.h, self.d_k).transpose(1, 0, 2)
        K = K.reshape(L, self.h, self.d_k).transpose(1, 0, 2)
        V = V.reshape(L, self.h, self.d_k).transpose(1, 0, 2)

        # ③ 每個頭各自做 attention
        heads, attns = [], []
        for i in range(self.h):
            o, a = scaled_dot_product_attention(Q[i], K[i], V[i], mask)
            heads.append(o)
            attns.append(a)

        # ④ 把 h 個頭接回來 (h, L, d_k) → (L, d_model)，再過 W_o
        concat = np.concatenate(heads, axis=-1)
        return concat @ self.W_o, np.array(attns)


mha = MultiHeadAttention(d_model=64, num_heads=8)
X_seq = np.random.randn(10, 64)
out, attns = mha(X_seq)

print("輸入形狀:", X_seq.shape)
print("輸出形狀:", out.shape, "  ← 和輸入一樣（這樣才能疊很多層）")
print("注意力形狀:", attns.shape, "  (頭數, 序列長, 序列長)")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(attns[i], cmap="Blues")
    ax.set_title(f"第 {i+1} 個頭", fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("8 個頭關注的模式都不一樣（這裡是隨機權重，訓練後會更有結構）", fontsize=13)
plt.tight_layout(); plt.show()

## 7. Positional Encoding

Attention 是加權平均，**加權平均沒有順序概念** ——
「狗咬人」和「人咬狗」在純 attention 眼中完全一樣。

所以必須把「你是第幾個字」編碼成向量，加進去。

In [ ]:
def positional_encoding(seq_len, d_model):
    pos = np.arange(seq_len)[:, None]
    i = np.arange(d_model)[None, :]
    angle = pos / np.power(10000, (2 * (i // 2)) / d_model)
    pe = np.where(i % 2 == 0, np.sin(angle), np.cos(angle))
    return pe


PE = positional_encoding(60, 64)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

im = axes[0].imshow(PE, cmap="RdBu", aspect="auto")
axes[0].set_xlabel("向量的第幾維"); axes[0].set_ylabel("第幾個位置")
axes[0].set_title("位置編碼矩陣（每一列是一個位置的獨特指紋）")
plt.colorbar(im, ax=axes[0])

for d in [0, 1, 4, 8, 16]:
    axes[1].plot(PE[:, d], label=f"第 {d} 維", lw=2)
axes[1].set_xlabel("位置"); axes[1].set_ylabel("值")
axes[1].set_title("不同維度是不同頻率的正弦波")
axes[1].legend(); axes[1].grid(alpha=.3)

plt.tight_layout(); plt.show()

In [ ]:
# 驗證：任兩個位置的編碼是不是真的不一樣？相鄰的是不是比較像？
sim = PE @ PE.T
sim = sim / np.linalg.norm(PE, axis=1, keepdims=True) / np.linalg.norm(PE, axis=1)

plt.figure(figsize=(7, 6))
plt.imshow(sim, cmap="viridis")
plt.colorbar(label="餘弦相似度")
plt.xlabel("位置"); plt.ylabel("位置")
plt.title("位置編碼之間的相似度")
plt.show()

print("👉 對角線最亮（自己跟自己最像），往兩側逐漸變暗。")
print("   代表「相鄰位置的編碼比較像」—— 這正是我們要的性質。")

## 8. 組裝一個完整的 Transformer Block

```
輸入 x
  ↓
① Multi-Head Self-Attention
② 殘差連接 + LayerNorm     ← x + attention(x)
③ Feed-Forward（放大 4 倍再縮回來）
④ 殘差連接 + LayerNorm
  ↓
輸出（形狀跟輸入一樣，所以可以一直疊）
```

In [ ]:
def layer_norm(x, eps=1e-6):
    """在「特徵」這個維度上標準化（注意：不是在 batch 維度，那是 BatchNorm）"""
    mean = x.mean(axis=-1, keepdims=True)
    std = x.std(axis=-1, keepdims=True)
    return (x - mean) / (std + eps)


class TransformerBlock:
    def __init__(self, d_model=64, num_heads=8, d_ff=256, seed=42):
        rng = np.random.default_rng(seed)
        self.mha = MultiHeadAttention(d_model, num_heads, seed)
        # Feed-Forward：先放大 4 倍，過 ReLU，再縮回來
        self.W1 = rng.normal(0, 1 / np.sqrt(d_model), (d_model, d_ff))
        self.b1 = np.zeros(d_ff)
        self.W2 = rng.normal(0, 1 / np.sqrt(d_ff), (d_ff, d_model))
        self.b2 = np.zeros(d_model)

    def __call__(self, x, mask=None):
        # ① + ② 注意力 + 殘差 + LayerNorm
        attn_out, attn_w = self.mha(x, mask)
        x = layer_norm(x + attn_out)               # ← 那個 "+ x" 就是殘差連接

        # ③ + ④ Feed-Forward + 殘差 + LayerNorm
        ff = np.maximum(0, x @ self.W1 + self.b1) @ self.W2 + self.b2
        x = layer_norm(x + ff)

        return x, attn_w


seq_len, d_model = 12, 64
x = np.random.randn(seq_len, d_model) + positional_encoding(seq_len, d_model)

block = TransformerBlock(d_model=d_model)
mask = np.tril(np.ones((seq_len, seq_len)))        # GPT 式的因果遮罩

out, attn_w = block(x, mask)

print("輸入形狀:", x.shape)
print("輸出形狀:", out.shape, " ← 完全一樣，所以可以疊 N 層（GPT-3 疊了 96 層）")
print()
print("輸出每一列的平均 ≈ 0：", np.abs(out.mean(axis=-1)).max() < 1e-6)
print("輸出每一列的標準差 ≈ 1：", np.abs(out.std(axis=-1) - 1).max() < 1e-3)
print("（LayerNorm 生效了）")

In [ ]:
# 疊 4 層看看
x_multi = x.copy()
blocks = [TransformerBlock(d_model=d_model, seed=i) for i in range(4)]

fig, axes = plt.subplots(1, 4, figsize=(17, 4.2))
for i, (blk, ax) in enumerate(zip(blocks, axes)):
    x_multi, aw = blk(x_multi, mask)
    ax.imshow(aw[0], cmap="Blues")           # 只看第 1 個頭
    ax.set_title(f"第 {i+1} 層的注意力")
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle("疊 4 層 Transformer Block（每層的注意力模式不同）")
plt.tight_layout(); plt.show()

print("最終輸出形狀:", x_multi.shape)

## 9. 複雜度實測：為什麼長文本這麼貴

Self-Attention 的複雜度是 **O(n² · d)** —— 序列長度加倍，計算量變 4 倍。

In [ ]:
print(f"{'序列長度':>10}{'注意力矩陣大小':>18}{'記憶體(float32)':>18}")
print("-" * 48)
for n in [128, 512, 2048, 8192, 32768, 131072]:
    cells = n * n
    mb = cells * 4 / 1024**2
    unit = f"{mb:,.0f} MB" if mb < 1024 else f"{mb/1024:,.1f} GB"
    print(f"{n:>10,}{cells:>18,}{unit:>18}")

print()
print("👉 序列 13 萬 token（約一本書）時，光是「一個頭、一層」的注意力矩陣")
print("   就要 64 GB。這就是長文本模型的根本瓶頸。")
print()
print("💡 解法方向：FlashAttention（分塊計算，不存整個矩陣）、")
print("   稀疏注意力（Longformer）、線性注意力、狀態空間模型（Mamba）。")

---

## 🎯 動手改改看

1. **改第 3 格的詞向量**，加入「老鼠」和「牠」，
   並設計向量讓「牠」的 Q 會關注動物。觀察注意力矩陣。

2. **把第 6 格的 `num_heads` 從 8 改成 1 和 16**，
   看注意力模式的差異。

3. **拿掉第 8 格的殘差連接**（改成 `x = layer_norm(attn_out)`），
   然後疊 10 層，看輸出的數值分布怎麼退化。
   → 這就是為什麼需要殘差連接。

4. **拿掉位置編碼**，把第 8 格輸入序列的順序打亂，
   驗證輸出是不是「跟順序無關」（只是行順序跟著換）。
   → 親自證明 attention 本身沒有順序概念。

5. **實作 Cross-Attention**：讓 Q 來自序列 A、K 和 V 來自序列 B。
   這是機器翻譯的核心。

## 📝 恭喜，四份 Notebook 完成

**接下來該做的**：
1. 回教學 App 的「💼 面試題庫」，把必考 15 題出聲講一次
2. 找一份真實資料（Kaggle 或 data.gov.tw），把 Notebook 01 的流程重跑一次
3. 把成果整理成 GitHub repo，寫好 README 的五個段落